# Phase 4-A：NII 驅動策略回測

**策略規則**：
- **進場**：`detect_phase()` 從「冷卻」→「預熱」
- **出場**：`detect_phase()` 進入「降溫」；或持有超過 `max_hold_days`；或停損

**核心發現**（2024-2026 CoWoS 回測）：
- 勝率 75%，但策略 +48.6% 遠落後買持 +137.7%
- 原因：AI 超導體 2 年牛市中，「預熱訊號」讓策略頻繁空倉
- **結論**：NII 最大價值是主題篩選（哪個題材值得關注），不是精確擇時
- **「降溫」出場訊號最有效**：最後幾次交易 +20~22%

In [ ]:
import sys
import logging
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)s — %(message)s',
                    datefmt='%H:%M:%S')
logging.getLogger('matplotlib').setLevel(logging.WARNING)

print('環境準備完成')

## 1. 單主題回測

In [ ]:
from src.analyzers.backtest import run_backtest

# === 設定回測參數 ===
TOPIC         = "cowos"   # 要回測的主題
MAX_HOLD_DAYS = 30        # 最大持有天數
SLOPE_WINDOW  = 14        # NII 斜率計算視窗（天）
STOP_LOSS     = -0.12     # 停損 -12%
COOL_TO_WARM  = True      # True=只在冷→熱轉換時買入（更保守）

result = run_backtest(
    topic             = TOPIC,
    max_hold_days     = MAX_HOLD_DAYS,
    slope_window      = SLOPE_WINDOW,
    stop_loss         = STOP_LOSS,
    cool_to_warm_only = COOL_TO_WARM,
    save_chart        = True,
)

result.print_summary()

In [ ]:
# 顯示回測圖（讀取已存的 PNG）
from IPython.display import Image
Image(str(ROOT / 'reports' / f'backtest_{TOPIC}.png'))

In [ ]:
# 所有交易明細
import pandas as pd

trades_data = [{
    '進場日': t.entry_date.date(),
    '出場日': t.exit_date.date(),
    '持有天': t.hold_days,
    '報酬': f"{t.ret*100:+.1f}%",
    '出場原因': t.exit_reason,
} for t in result.trades]

pd.DataFrame(trades_data)

## 2. 參數敏感度分析

In [ ]:
import pandas as pd
from src.analyzers.backtest import run_backtest, _sharpe, _max_drawdown

param_grid = [
    {"max_hold_days": 30,  "cool_to_warm_only": True,  "stop_loss": -0.12},
    {"max_hold_days": 60,  "cool_to_warm_only": True,  "stop_loss": -0.12},
    {"max_hold_days": 90,  "cool_to_warm_only": True,  "stop_loss": -0.15},
    {"max_hold_days": 30,  "cool_to_warm_only": False, "stop_loss": -0.12},
    {"max_hold_days": 60,  "cool_to_warm_only": False, "stop_loss": -0.12},
]

rows = []
for p in param_grid:
    r = run_backtest(TOPIC, save_chart=False, **p)
    s = r.summary()
    rows.append({
        'max_hold': p['max_hold_days'],
        'cool→warm only': p['cool_to_warm_only'],
        'stop_loss': p['stop_loss'],
        '交易次數': s['交易次數'],
        '勝率': s['勝率'],
        '策略報酬': s['策略累積報酬'],
        '買持報酬': s['買持累積報酬'],
        'Sharpe': s['年化Sharpe'],
        '最大回撤': s['最大回撤'],
    })

pd.DataFrame(rows)

## 3. 多主題比較

⚠️ 被動元件 / 光通訊只有 91 天資料（2026-01-01~），樣本過小，僅供參考。

In [ ]:
from src.analyzers.backtest import run_backtest
from pathlib import Path

# 找出哪些主題已有完整資料
processed_dir = ROOT / 'data' / 'processed'
available = [p.stem.replace('_nii', '') for p in processed_dir.glob('*_nii.parquet')]
print('可回測主題：', available)

rows = []
for topic in available:
    try:
        r = run_backtest(topic, max_hold_days=30, save_chart=False)
        s = r.summary()
        s['主題'] = topic
        rows.append(s)
    except Exception as e:
        print(f'{topic}: 跳過 ({e})')

if rows:
    pd.DataFrame(rows)[['主題','交易次數','勝率','策略累積報酬','買持累積報酬','超額報酬','年化Sharpe','最大回撤']]

## 4. NII 訊號解讀框架

基於回測結果，建議以下訊號使用方式：

| 訊號 | 含義 | 建議行動 |
|------|------|----------|
| 冷卻→預熱 | 訊息熱度剛開始上升 | ✅ 開始關注此主題；此時股價可能尚未反應 |
| 預熱→發燒 | 市場熱度爆發 | ⚠️ 訊息高峰期；股價可能已充分反應 |
| 發燒→降溫 | 訊息熱度開始消退 | 🔴 可能是出場時機；考慮減碼 |
| 降溫→冷卻 | 題材熱度歸零 | 🔄 等待下一輪醞釀 |

**最強複合訊號**：NII 預熱 + 外資買超 → 法人同步認可，最佳進場條件

In [ ]:
# 查看 cowos 的相位歷史
import pandas as pd

phases = result.phase_series.value_counts()
print('CoWoS 2024-2026 相位分布：')
print(phases.to_string())
print(f'\n預熱天數比例：{phases.get("預熱", 0) / phases.sum() * 100:.1f}%')
print(f'發燒天數比例：{phases.get("發燒", 0) / phases.sum() * 100:.1f}%')

## 5. 籌碼複合訊號驗證

結合 NII 預熱 + 外資買超，驗證複合訊號的有效性。

（需要先執行 `src/collectors/chips.py` 採集籌碼資料）

In [ ]:
from pathlib import Path

# 檢查哪些主題有籌碼資料
chips_dir = ROOT / 'data' / 'raw' / 'chips'
topics_with_chips = [p.stem.replace('_institutional', '') 
                      for p in chips_dir.glob('*_institutional.parquet')]
print('有籌碼資料的主題：', topics_with_chips)

if topics_with_chips:
    from src.processors.chips_signal import build_chips_signal
    topic_c = topics_with_chips[0]
    chips_df = build_chips_signal(topic_c, save=False)
    print(f'\n{topic_c} 籌碼訊號（最近 10 天）：')
    print(chips_df.tail(10).to_string())